# Fraud Detection - Machine Learning Modeling

## Project Overview
This notebook implements machine learning models for fraud detection using the IEEE-CIS dataset. Building on insights from our comprehensive EDA, we focus on creating robust, interpretable models that can effectively identify fraudulent transactions while minimizing false positives.

## Modeling Strategy
Based on our EDA findings, we will:
1. **Use the 20 most predictive features** identified through correlation and business logic analysis
2. **Handle severe class imbalance** (3.5% fraud rate) using appropriate sampling and evaluation techniques
3. **Compare multiple algorithms** to find the best performing approach
4. **Track all experiments** using MLflow for reproducibility and comparison
5. **Focus on precision-recall metrics** rather than accuracy due to class imbalance

## Key EDA Insights Applied
- **V-features (V45, V44, V86, V87, V52)** are the strongest predictors (0.24-0.28 correlation)
- **Categorical features** (ProductCD, card4, P_emaildomain) provide clear fraud patterns
- **Email domains** like mail.com show 19% fraud rates (5.4x higher than average)
- **Device/location features** (D-features) offer moderate but valuable signals
- **Transaction amount** alone is a weak predictor (0.011 correlation)

## Success Metrics
- **Primary**: Precision-Recall AUC (better for imbalanced data)
- **Secondary**: F1-Score, Recall@95% Precision
- **Business**: Fraud detection rate vs false positive rate

## 1. Environment Setup & Configuration

In [5]:
# Import necessary libraries
import pandas as pd
import numpy as np
import os
import warnings
import random
import joblib
import pickle

# Data preprocessing
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Machine learning models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
import lightgbm as lgb

# Evaluation metrics
from sklearn.metrics import (
    classification_report, confusion_matrix, 
    precision_recall_curve, roc_curve, auc,
    precision_score, recall_score, f1_score,
    accuracy_score, log_loss, roc_auc_score,
    average_precision_score
)

# MLflow for experiment tracking
import mlflow
import mlflow.sklearn
import mlflow.xgboost
import mlflow.lightgbm

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

print("All libraries imported successfully!")
print(f"Python version: {os.sys.version}")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")
print(f"MLflow version: {mlflow.__version__}")
print(f"XGBoost version: {xgb.__version__}")
print(f"LightGBM version: {lgb.__version__}")

All libraries imported successfully!
Python version: 3.13.7 (tags/v3.13.7:bcee1c3, Aug 14 2025, 14:15:11) [MSC v.1944 64 bit (AMD64)]
Pandas version: 2.3.3
NumPy version: 2.3.3
MLflow version: 3.5.0
XGBoost version: 3.0.5
LightGBM version: 4.6.0


In [6]:
# Configure global settings and random seeds for reproducibility
RANDOM_STATE = 42
TEST_SIZE = 0.2
CV_FOLDS = 5

# Set all random seeds for reproducibility
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

# Configure pandas and matplotlib for better display
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 50)
pd.set_option('display.width', 1000)

plt.style.use('default')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

# Create configuration dictionary for easy access
CONFIG = {
    'random_state': RANDOM_STATE,
    'test_size': TEST_SIZE,
    'cv_folds': CV_FOLDS,
    'mlflow_tracking_uri': '../mlruns',
    'experiment_name': 'fraud-detection-ieee-cis',
    'data_path': '../data/',
    'models_path': '../models/',
    'results_path': '../results/'
}

print("Configuration settings established:")
print(f"  Random State: {CONFIG['random_state']}")
print(f"  Test Size: {CONFIG['test_size']}")
print(f"  CV Folds: {CONFIG['cv_folds']}")
print(f"  MLflow Tracking URI: {CONFIG['mlflow_tracking_uri']}")
print(f"  Experiment Name: {CONFIG['experiment_name']}")

# Create directories if they don't exist
for path_key in ['models_path', 'results_path']:
    path = CONFIG[path_key]
    if not os.path.exists(path):
        os.makedirs(path)
        print(f"  Created directory: {path}")
    else:
        print(f"  Directory exists: {path}")

Configuration settings established:
  Random State: 42
  Test Size: 0.2
  CV Folds: 5
  MLflow Tracking URI: ../mlruns
  Experiment Name: fraud-detection-ieee-cis
  Directory exists: ../models/
  Directory exists: ../results/


In [7]:
# Configure MLflow for experiment tracking
# Set tracking URI to local directory
mlflow.set_tracking_uri(CONFIG['mlflow_tracking_uri'])

# Create or set experiment
experiment_name = CONFIG['experiment_name']

# Check if experiment exists, create if not
try:
    experiment = mlflow.get_experiment_by_name(experiment_name)
    if experiment is None:
        experiment_id = mlflow.create_experiment(experiment_name)
        print(f"Created new MLflow experiment: '{experiment_name}' (ID: {experiment_id})")
    else: 
        experiment_id = experiment.experiment_id
        print(f"Experiment already exists: '{experiment_name}' (ID: {experiment_id})")

        # Set experiment as active
        mlflow.set_experiment(experiment_name)
except Exception as e:
    print(f"Error setting up MLflow experiment: {e}")
    print("Continuing without MLflow experiment tracking...")

# Display MLflow configuration
print(f"\nMLflow Configuration:")
print(f"  Tracking URI: {mlflow.get_tracking_uri()}")
print(f"  Active Experiment: {mlflow.get_experiment_by_name(experiment_name).name if mlflow.get_experiment_by_name(experiment_name) else 'None'}")
print(f"  Artifact Location: {mlflow.get_experiment_by_name(experiment_name).artifact_location if mlflow.get_experiment_by_name(experiment_name) else 'None'}")

print(f"\nMLflow experiment tracking configured successfully!")

Created new MLflow experiment: 'fraud-detection-ieee-cis' (ID: 999411205517379395)

MLflow Configuration:
  Tracking URI: ../mlruns
  Active Experiment: fraud-detection-ieee-cis
  Artifact Location: file:c:/Users/Admin/Documents/ML_Engineering/Fraud_Detection/notebooks/../mlruns/999411205517379395

MLflow experiment tracking configured successfully!


In [8]:
# Load training and test datasets
print("Loading datasets...")

# Load transaction data
train_transaction = pd.read_csv('../data/train_transaction.csv')
print(f"Train transaction data loaded: {train_transaction.shape}")

train_identity = pd.read_csv('../data/train_identity.csv')
print(f"Train identity data loaded: {train_identity.shape}")

test_transaction = pd.read_csv('../data/test_transaction.csv')
print(f"Test transaction data loaded: {test_transaction.shape}")

test_identity = pd.read_csv('../data/test_identity.csv')
print(f"Test identity data loaded: {test_identity.shape}")

# Merge transaction and identity data
print("\nMerging transaction and identity data...")

# Merge training data (left join to keep all transactions)
df_train = train_transaction.merge(train_identity, on='TransactionID', how='left')
print(f"Training data merged: {df_train.shape}")

# Merge test data (left join to keep all transactions)
df_test = test_transaction.merge(test_identity, on='TransactionID', how='left')
print(f"Test data merged: {df_test.shape}")

# Display basic information
print(f"\nDataset Summary:")
print(f"  Training samples: {len(df_train):,}")
print(f"  Test samples: {len(df_test):,}")
print(f"  Total features: {df_train.shape[1]:,}")
print(f"  Target variable: isFraud")

# Check fraud distribution in training data
fraud_count = df_train['isFraud'].sum()
fraud_rate = fraud_count / len(df_train) * 100
print(f"\nFraud Distribution:")
print(f"  Fraudulent transactions: {fraud_count:,} ({fraud_rate:.2f}%)")
print(f"  Legitimate transactions: {len(df_train) - fraud_count:,} ({100-fraud_rate:.2f}%)")
print(f"  Class imbalance ratio: {(len(df_train) - fraud_count) / fraud_count:.1f}:1")

print(f"\nAll datasets loaded and merged successfully!")

Loading datasets...


MemoryError: Unable to allocate 1.65 GiB for an array with shape (376, 590540) and data type float64

### Why These Setup Choices Matter

#### **MLflow Experiment Tracking**
We use MLflow because:
- **Reproducibility**: Every model run is logged with parameters, metrics, and artifacts
- **Comparison**: Easy comparison between different algorithms and hyperparameters
- **Collaboration**: Team members can view and compare experiments
- **Model Registry**: Track model versions and deployment status
- **Artifact Storage**: Save models, plots, and preprocessors automatically

#### **Random Seeds Ensure Reproducibility**
Setting `RANDOM_STATE = 42` across all components ensures:
- **Consistent train/test splits**: Same data division every time
- **Reproducible model training**: Same initialization and sampling
- **Comparable results**: Fair comparison between algorithm runs
- **Debugging capability**: Ability to recreate exact same results
- **Scientific rigor**: Essential for peer review and production deployment

#### **Configuration Dictionary Benefits**
Using `CONFIG` dictionary provides:
- **Single source of truth**: All settings in one place
- **Easy experimentation**: Change parameters without hunting through code
- **Documentation**: Clear visibility of all experimental settings
- **Maintainability**: Easier to update and modify parameters

#### **Class Imbalance Challenge**
Our 27.6:1 imbalance ratio means:
- **Accuracy is misleading**: 96.5% accuracy by always predicting "legitimate"
- **Special techniques needed**: SMOTE, undersampling, class weights
- **Metric focus**: Precision-Recall AUC over standard accuracy
- **Business impact**: False negatives (missed fraud) very costly

#### **Next Steps Overview**
With 434 raw features and 590K samples, we will:
1. **Feature Selection**: Reduce to 20 most predictive features (from EDA)
2. **Preprocessing**: Handle missing values and encode categoricals
3. **Sampling**: Address class imbalance using proven techniques
4. **Modeling**: Compare 4 algorithms (LogisticRegression, RandomForest, XGBoost, LightGBM)
5. **Evaluation**: Focus on business-relevant metrics (Precision-Recall AUC, F1-Score)

This systematic approach ensures our models are not just accurate, but practical for real-world fraud detection deployment.